# 01 · Zero-shot Baseline — Qwen2.5-VL (4-bit) trên ViVQA

**Mục tiêu notebook:** nạp baseline VLM (Qwen2.5-VL-3B) ở chế độ 4-bit và đo
**EM · VQA-Accuracy · ANLS** theo kiểu *zero-shot* (chưa fine-tune) trên ViVQA.

Đây là **con số baseline** để mọi cải tiến (LoRA, OCR, RAG) ở các bước sau so sánh với.

> ⚠️ **Bắt buộc chạy trên GPU** (Colab Pro / Kaggle). `bitsandbytes` không chạy trên CPU.
> Vào `Runtime > Change runtime type > GPU` trước khi chạy.

**Luồng:** cài môi trường → nạp code dự án → (tùy chọn) demo 2 ảnh để test pipeline →
trỏ ViVQA thật → nạp model → chạy eval → lưu experiment-log.

## 0. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 1. Cài đặt thư viện

Pin phiên bản `transformers` đủ mới để có lớp `Qwen2_5_VLForConditionalGeneration`.

In [ ]:
!pip -q install "transformers>=4.49.0" accelerate peft bitsandbytes \
    qwen-vl-utils pyyaml pillow python-Levenshtein 2>/dev/null
print('Đã cài xong.')

## 2. Nạp code dự án (`src/` + `configs/`)

**Chọn 1 trong 2 cách** bên dưới. Mặc định dùng cách A (git clone).

- **Cách A — Git clone:** sửa `REPO_URL` thành repo GitHub của nhóm.
- **Cách B — Google Drive:** bỏ comment phần mount, trỏ `PROJECT_DIR` tới thư mục đã upload.

In [ ]:
import os, sys

# ---- Cách A: Git clone (khuyến nghị) ----
REPO_URL = 'https://github.com/<YOUR_TEAM>/ViVOA-VLM.git'  # TODO: sửa URL
PROJECT_DIR = '/content/ViVOA-VLM'
if not os.path.exists(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR

# ---- Cách B: Google Drive (bỏ comment nếu dùng) ----
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = '/content/drive/MyDrive/ViVOA-VLM'

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)  # để 'import src...' hoạt động
print('Thư mục dự án:', os.getcwd())
print('Có src/ ?', os.path.isdir('src'), '| Có configs/ ?', os.path.isdir('configs'))

## 3. Đọc config

Toàn bộ tham số run lấy từ `configs/qwen_lora.yaml`. Zero-shot nên ta ép `mode='zero_shot'`.

In [ ]:
import yaml
with open('configs/qwen_lora.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# Ép chế độ zero-shot cho baseline này + đặt tên run rõ ràng
cfg['prompting']['mode'] = 'zero_shot'
cfg['run']['name'] = 'qwen2_5_vl_3b_zeroshot_vivqa'
print('Backbone :', cfg['model']['model_id'])
print('QLoRA    :', cfg['quantization']['enabled'])
print('Mode     :', cfg['prompting']['mode'])

## 4. (Tùy chọn) DEMO 2 ảnh — kiểm tra pipeline trước khi dùng dữ liệu thật

Đặt `USE_DEMO = True` để tải 2 ảnh COCO công khai và chạy thử end-to-end.
Khi đã chắc pipeline chạy, đặt `USE_DEMO = False` và trỏ ViVQA thật ở mục 5.

In [ ]:
USE_DEMO = True

if USE_DEMO:
    import urllib.request, json
    os.makedirs('data/demo/images', exist_ok=True)
    demo_imgs = {
        'cat.jpg': 'http://images.cocodataset.org/val2017/000000039769.jpg',
        'bus.jpg': 'http://images.cocodataset.org/val2017/000000000285.jpg',
    }
    for name, url in demo_imgs.items():
        p = f'data/demo/images/{name}'
        if not os.path.exists(p):
            urllib.request.urlretrieve(url, p)
    demo = [
        {'question_id': 'd1', 'question': 'Trong ảnh có con vật gì?',
         'answers': ['con mèo', 'mèo'], 'image': 'cat.jpg', 'question_type': 'what'},
        {'question_id': 'd2', 'question': 'Trong ảnh có mấy con gấu?',
         'answers': ['hai', '2'], 'image': 'bus.jpg', 'question_type': 'counting'},
    ]
    with open('data/demo/demo.json', 'w', encoding='utf-8') as f:
        json.dump(demo, f, ensure_ascii=False)
    print('Đã tạo demo 2 ảnh.')

## 5. Nạp dữ liệu ViVQA (hoặc demo)

**Với ViVQA thật:** đặt file JSON + ảnh vào `data/vivqa/`, rồi chỉnh `field_map`
cho khớp tên trường của file (xem `src/data/vivqa_dataset.py`). Nếu ViVQA chỉ cho
`image_id` kiểu COCO, đặt `image_pattern='COCO_val2014_{:012d}.jpg'`.

In [ ]:
from src.data.vivqa_dataset import load_vivqa

if USE_DEMO:
    samples = load_vivqa('data/demo/demo.json', image_dir='data/demo/images')
else:
    samples = load_vivqa(
        data_path=cfg['data']['test_path'],      # vd 'data/vivqa/test.json'
        image_dir=cfg['data']['image_dir'],
        field_map={'question': 'question', 'answer': 'answer', 'image': 'image'},  # TODO khớp file
        image_pattern=None,                       # hoặc 'COCO_val2014_{:012d}.jpg'
        max_samples=cfg['data'].get('max_samples'),
    )
print(f'Số mẫu: {len(samples)}')
print('Mẫu đầu:', samples[0])

## 6. Nạp model Qwen2.5-VL (4-bit)

`load_vlm_for_inference` nạp checkpoint gốc ở 4-bit — **không** gắn LoRA (vì đây là
zero-shot baseline). Lần đầu sẽ tải ~vài GB weight, hãy kiên nhẫn.

In [ ]:
import torch, random, numpy as np
# Cố định seed (PDF bắt buộc)
seed = cfg['run']['seed']
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

from src.models.vlm_loader import load_vlm_for_inference
model, processor = load_vlm_for_inference(cfg)
print('Đã nạp model trên:', model.device)

## 7. Chạy đánh giá zero-shot

`run_evaluation` sinh câu trả lời cho từng mẫu → tính EM/VQA-Acc/ANLS (tổng thể +
tách theo question type) → kèm metadata reproducibility.

In [ ]:
from src.eval.run_eval import run_evaluation, save_results

result = run_evaluation(model, processor, samples, cfg, verbose_every=20)
run_dir = save_results(result, cfg['run']['output_dir'])
print('Đã lưu kết quả tại:', run_dir)

## 8. Xem kết quả

In [ ]:
import pandas as pd
m = result['metrics']

# Bảng metric tổng thể (chỉ các khóa KHÔNG chứa '__')
overall = {k: v for k, v in m.items() if '__' not in k}
print('=== METRIC TỔNG THỂ ===')
display(pd.DataFrame([overall]).T.rename(columns={0: 'score'}))

# Bảng theo question type (phục vụ error analysis)
rows = {}
for k, v in m.items():
    if '__' in k:
        met, qt = k.split('__', 1)
        rows.setdefault(qt, {})[met] = v
if rows:
    print('=== THEO LOẠI CÂU HỎI ===')
    display(pd.DataFrame(rows).T)

print('=== META (reproducibility) ===')
print(json.dumps(result['meta'], ensure_ascii=False, indent=2))

## 9. Xem vài dự đoán cụ thể (định tính)

In [ ]:
for r in result['predictions'][:5]:
    print(f"Q: {r['question']}")
    print(f"   pred = {r['prediction']!r}")
    print(f"   gt   = {r['answers']}")
    print('-' * 60)

---
### ✅ Xong Bước 2

Bạn đã có **số baseline zero-shot** + file `experiments/runs/.../metrics.json`.

**Bước 3 tiếp theo:** script cache OCR (PaddleOCR/VietOCR) cho ViTextVQA → bật
`mode='ocr'` và chạy lại chính notebook này để có dòng ablation *+OCR*.